# Silver Cleansing: Food Inspections (Chicago & Dallas)

Pipeline parameters

In [0]:
dbutils.widgets.text("catalog_name", "workspace")
dbutils.widgets.text("schema_name", "foodlens_bronze")
dbutils.widgets.text("silver_schema", "foodlens_silver")

catalog_name  = dbutils.widgets.get("catalog_name")
schema_name   = dbutils.widgets.get("schema_name")
silver_schema = dbutils.widgets.get("silver_schema")

print("=" * 50)
print("PIPELINE PARAMETERS")
print("=" * 50)
print(f"Catalog       : {catalog_name}")
print(f"Bronze Schema : {schema_name}")
print(f"Silver Schema : {silver_schema}")
print("=" * 50)

In [0]:
%pip install databricks-labs-dqx

In [0]:
dbutils.library.restartPython()

Re-read parameters after restart

In [0]:
catalog_name  = dbutils.widgets.get("catalog_name")
schema_name   = dbutils.widgets.get("schema_name")
silver_schema = dbutils.widgets.get("silver_schema")

from pyspark.sql.functions import (
    col, lit, trim, upper, lower, when, regexp_extract, regexp_replace,
    split, explode, posexplode, size, expr, count, length, substring,
    monotonically_increasing_id, current_timestamp, concat_ws, array,
    sha2, concat
)
from pyspark.sql.types import IntegerType, DoubleType, StringType
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient
import re

wc  = WorkspaceClient()
dqe = DQEngine(workspace_client=wc, spark=spark)

print(f"Parameters reloaded: {catalog_name}.{schema_name} -> {catalog_name}.{silver_schema}")
print("DQEngine initialized")

Load Bronze tables

In [0]:
chicago_raw = spark.table(f"{catalog_name}.{schema_name}.chicago_inspections")
dallas_raw  = spark.table(f"{catalog_name}.{schema_name}.dallas_inspections")

print(f"Chicago Bronze: {chicago_raw.count():,} rows, {len(chicago_raw.columns)} cols")
print(f"Dallas Bronze:  {dallas_raw.count():,} rows, {len(dallas_raw.columns)} cols")

Parse Chicago violations string into code, description, comment

In [0]:
# Chicago violations are pipe delimited: "code. DESCRIPTION ... Comments: ..." separated by " | "

from pyspark.sql.functions import posexplode, trim, regexp_extract

chi_violations_exploded = (
    chicago_raw
    .filter(col("Violations").isNotNull())
    .select(
        col("Inspection ID"),
        posexplode(split(col("Violations"), " \\| ")).alias("violation_seq", "raw_violation")
    )
)

chi_violations_parsed = (
    chi_violations_exploded
    .withColumn("raw_violation", trim(col("raw_violation")))
    .withColumn("violation_code",
        regexp_extract(col("raw_violation"), r"^(\d+)\.", 1))
    .withColumn("violation_description",
        trim(regexp_extract(col("raw_violation"), r"^\d+\.\s*([^\-]+?)(?:\s*-\s*Comments:|$)", 1)))
    .withColumn("violation_comment",
        trim(regexp_extract(col("raw_violation"), r"Comments:\s*(.*)", 1)))
    .filter(col("violation_code") != "")
)

chi_violations_parsed.select("Inspection ID", "violation_seq", "violation_code", "violation_description", "violation_comment").show(10, truncate=80)
print(f"Chicago violations parsed: {chi_violations_parsed.count():,} rows from {chi_violations_exploded.count():,} exploded chunks")

Unpivot Dallas 25 violation slots to long format, extract violation_code from description

In [0]:
# Dallas has 25 slots: Violation Description/Points/Detail/Memo for each slot
# Unpivot all 25 into long format rows using SQL UNION ALL (serverless compatible)

dallas_core_cols = [
    "Restaurant Name", "Inspection Type", "Inspection Date", "Inspection Score",
    "Street Number", "Street Name", "Street Direction", "Street Type", "Street Unit",
    "Street Address", "Zip Code", "Inspection Month", "Inspection Year", "Lat Long Location",
    "_source_file", "_load_timestamp", "_source_city"
]

dallas_raw.createOrReplaceTempView("dallas_bronze")

available_core = [c for c in dallas_core_cols if c in dallas_raw.columns]
core_select = ", ".join([f"`{c}`" for c in available_core])

union_parts = []
for i in range(1, 26):
    desc_col = f"Violation Description - {i}"
    pts_col  = f"Violation Points - {i}"
    det_col  = f"Violation Detail - {i}"
    memo_col = f"Violation Memo - {i}"

    if memo_col not in dallas_raw.columns:
        memo_col_alt = f"Violation  Memo - {i}"
        if memo_col_alt in dallas_raw.columns:
            memo_col = memo_col_alt

    if desc_col in dallas_raw.columns:
        pts_expr = f"CAST(`{pts_col}` AS INT)" if pts_col in dallas_raw.columns else "CAST(NULL AS INT)"
        det_expr = f"`{det_col}`" if det_col in dallas_raw.columns else "CAST(NULL AS STRING)"
        mem_expr = f"`{memo_col}`" if memo_col in dallas_raw.columns else "CAST(NULL AS STRING)"

        part = f"""
            SELECT {core_select},
                   {i} AS violation_seq,
                   `{desc_col}` AS violation_description,
                   {pts_expr} AS violation_points,
                   {det_expr} AS violation_detail,
                   {mem_expr} AS violation_memo
            FROM dallas_bronze
            WHERE `{desc_col}` IS NOT NULL"""
        union_parts.append(part)

full_sql = "\nUNION ALL\n".join(union_parts)
dallas_violations_long = spark.sql(full_sql)

# FIX V-02: extract violation_code from description (e.g. "*01 Cooling..." -> code="01")
dallas_violations_long = (
    dallas_violations_long
    .withColumn("violation_code",
        regexp_extract(col("violation_description"), r"^\*(\d+)", 1))
    .withColumn("violation_description",
        trim(regexp_replace(col("violation_description"), r"^\*\d+\s*", "")))
)

print(f"Dallas violations unpivoted: {dallas_violations_long.count():,} rows")
dallas_violations_long.select("Restaurant Name", "Inspection Date", "violation_seq", "violation_code", "violation_description", "violation_points").show(10, truncate=60)

Generate a unique inspection_id for Dallas (no natural key exists)

In [0]:
# FIX V-03: Dallas has no unique inspection ID
# Generate one using sha2 hash of all core inspection fields

dallas_raw_with_id = (
    dallas_raw
    .withColumn("inspection_id",
        sha2(concat(
            col("Restaurant Name"),
            col("Inspection Date").cast("string"),
            col("Inspection Score").cast("string"),
            col("Inspection Type"),
            col("Street Address")
        ), 256)
    )
)

# also add inspection_id to the violations long table
dallas_violations_long = (
    dallas_violations_long
    .withColumn("inspection_id",
        sha2(concat(
            col("Restaurant Name"),
            col("Inspection Date").cast("string"),
            col("Inspection Score").cast("string"),
            col("Inspection Type"),
            col("Street Address")
        ), 256)
    )
)

dups = dallas_raw_with_id.groupBy("inspection_id").count().filter(col("count") > 1).count()
print(f"Dallas inspection_id generated: {dallas_raw_with_id.count():,} rows, {dups} duplicate IDs")
print("Sample:")
dallas_raw_with_id.select("inspection_id", "Restaurant Name", "Inspection Date", "Inspection Score").show(5, truncate=40)

Build Chicago Silver table with derived scores and violation counts

In [0]:
chi_violation_counts = (
    chi_violations_parsed
    .groupBy("Inspection ID")
    .agg(count("*").alias("violation_count"))
)

chicago_silver = (
    chicago_raw
    .withColumn("inspection_score", 
        when(col("Results") == "Pass", 90)
        .when(col("Results") == "Pass w/ Conditions", 80)
        .when(col("Results") == "Fail", 70)
        .when(col("Results") == "No Entry", 0)
        .otherwise(lit(None).cast("int"))
    )
    .withColumn("City", upper(trim(col("City"))))
    .withColumn("State", upper(trim(col("State"))))
    .withColumn("source_city", lit("Chicago"))
)

chicago_silver = chicago_silver.join(
    chi_violation_counts, on="Inspection ID", how="left"
).fillna(0, subset=["violation_count"])

# FIX V-01: add has_urgent_critical flag (check violation text for Urgent/Critical keywords)
chi_urgent = (
    chi_violations_parsed
    .filter(
        lower(col("violation_description")).contains("urgent") |
        lower(col("violation_description")).contains("critical") |
        lower(col("violation_comment")).contains("urgent") |
        lower(col("violation_comment")).contains("critical")
    )
    .select("Inspection ID")
    .distinct()
    .withColumn("has_urgent_critical", lit(True))
)

chicago_silver = chicago_silver.join(chi_urgent, on="Inspection ID", how="left") \
    .fillna(False, subset=["has_urgent_critical"])

print(f"Chicago Silver: {chicago_silver.count():,} rows")
print(f"Inspections with Urgent/Critical violations: {chi_urgent.count():,}")
chicago_silver.select("Inspection ID", "DBA Name", "Results", "inspection_score", "City", "violation_count", "has_urgent_critical").show(10, truncate=False)

Build Dallas Silver table with violation counts using unique inspection_id

In [0]:
# FIX V-03: use inspection_id for counting instead of (Name, Date, Score)
dal_violation_counts = (
    dallas_violations_long
    .groupBy("inspection_id")
    .agg(count("*").alias("violation_count"))
)

dallas_silver = (
    dallas_raw_with_id
    .select(
        "inspection_id",
        *[c for c in dallas_core_cols if c in dallas_raw.columns]
    )
    .withColumn("source_city", lit("Dallas"))
)

dallas_silver = dallas_silver.join(
    dal_violation_counts, on="inspection_id", how="left"
).fillna(0, subset=["violation_count"])

# FIX V-01: add has_urgent_critical flag for Dallas
dal_urgent = (
    dallas_violations_long
    .filter(
        lower(col("violation_description")).contains("urgent") |
        lower(col("violation_description")).contains("critical") |
        lower(col("violation_detail")).contains("urgent") |
        lower(col("violation_detail")).contains("critical") |
        lower(col("violation_memo")).contains("urgent") |
        lower(col("violation_memo")).contains("critical")
    )
    .select("inspection_id")
    .distinct()
    .withColumn("has_urgent_critical", lit(True))
)

dallas_silver = dallas_silver.join(dal_urgent, on="inspection_id", how="left") \
    .fillna(False, subset=["has_urgent_critical"])

print(f"Dallas Silver: {dallas_silver.count():,} rows")
print(f"Inspections with Urgent/Critical violations: {dal_urgent.count():,}")
dallas_silver.select("inspection_id", "Restaurant Name", "Inspection Date", "Inspection Score", "Zip Code", "violation_count", "has_urgent_critical").show(10, truncate=False)

DQX validation rules from assignment requirements

In [0]:
# Assignment validation rules
# All rules use sql_expression because DQX is_not_null can't handle column names with spaces

chicago_rules = [
    {"name": "restaurant_name_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`DBA Name` IS NOT NULL",
         "msg": "Restaurant name cannot be null"}}},

    {"name": "inspection_date_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Inspection Date` IS NOT NULL",
         "msg": "Inspection date cannot be null"}}},

    {"name": "inspection_type_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Inspection Type` IS NOT NULL",
         "msg": "Inspection type cannot be null"}}},

    {"name": "zip_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Zip` IS NOT NULL",
         "msg": "Zip code cannot be null"}}},

    {"name": "zip_valid_format", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "CAST(`Zip` AS STRING) RLIKE '^[0-9]{5}$'",
         "msg": "Chicago zip must be valid 5 digit format"}}},

    {"name": "results_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Results` IS NOT NULL",
         "msg": "Chicago inspection results cannot be null"}}},

    {"name": "has_at_least_1_violation", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "violation_count >= 1",
         "msg": "Every inspection must have at least 1 violation"}}},

    {"name": "pass_no_urgent_critical", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "NOT (Results = 'Pass' AND has_urgent_critical = true)",
         "msg": "Result cannot be PASS if violation contains Urgent/Critical"}}},
]

dallas_rules = [
    {"name": "restaurant_name_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Restaurant Name` IS NOT NULL",
         "msg": "Restaurant name cannot be null"}}},

    {"name": "inspection_date_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Inspection Date` IS NOT NULL",
         "msg": "Inspection date cannot be null"}}},

    {"name": "inspection_type_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Inspection Type` IS NOT NULL",
         "msg": "Inspection type cannot be null"}}},

    {"name": "zip_not_null", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Zip Code` IS NOT NULL",
         "msg": "Zip code cannot be null"}}},

    {"name": "zip_valid_format", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "CAST(`Zip Code` AS STRING) RLIKE '^[0-9]{5}(-[0-9]{4})?$'",
         "msg": "Zip code must be valid 5 or 9 digit format"}}},

    {"name": "score_not_over_100", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Inspection Score` <= 100",
         "msg": "Dallas violation score cannot exceed 100"}}},

    {"name": "score_not_negative", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "`Inspection Score` >= 0",
         "msg": "Dallas violation score cannot be negative"}}},

    {"name": "has_at_least_1_violation", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "violation_count >= 1",
         "msg": "Every inspection must have at least 1 violation"}}},

    {"name": "pass_no_urgent_critical", "criticality": "error",
     "check": {"function": "sql_expression", "arguments": {
         "expression": "NOT (`Inspection Score` >= 90 AND has_urgent_critical = true)",
         "msg": "High score with Urgent/Critical violations"}}},
]

print(f"Chicago rules: {len(chicago_rules)}")
print(f"Dallas rules:  {len(dallas_rules)}")
for r in chicago_rules:
    print(f"  [CHI] {r['name']} ({r['criticality']})")
for r in dallas_rules:
    print(f"  [DAL] {r['name']} ({r['criticality']})")

Run DQX validation on both cities

In [0]:
print("=" * 60)
print("CHICAGO DQX VALIDATION")
print("=" * 60)

chi_good, chi_bad = dqe.apply_checks_by_metadata_and_split(chicago_silver, chicago_rules)

chi_total   = chicago_silver.count()
chi_passed  = chi_good.count()
chi_failed  = chi_bad.count()

print(f"Total:       {chi_total:,}")
print(f"Passed:      {chi_passed:,}")
print(f"Quarantined: {chi_failed:,}")

if chi_failed > 0:
    print("\nSample quarantined records:")
    chi_bad.select("Inspection ID", "DBA Name", "Results", "violation_count", "_errors", "_warnings").show(10, truncate=80)

In [0]:
print("=" * 60)
print("DALLAS DQX VALIDATION")
print("=" * 60)

dal_good, dal_bad = dqe.apply_checks_by_metadata_and_split(dallas_silver, dallas_rules)

dal_total   = dallas_silver.count()
dal_passed  = dal_good.count()
dal_failed  = dal_bad.count()

print(f"Total:       {dal_total:,}")
print(f"Passed:      {dal_passed:,}")
print(f"Quarantined: {dal_failed:,}")

if dal_failed > 0:
    print("\nSample quarantined records:")
    dal_bad.select("inspection_id", "Restaurant Name", "Inspection Date", "Inspection Score", "violation_count", "_errors", "_warnings").show(10, truncate=80)

Deduplicate violations per inspection by content, not slot position

In [0]:
# deduplicate violations per inspection by content

chi_violations_deduped = chi_violations_parsed.dropDuplicates(
    ["Inspection ID", "violation_code"]
)
print(f"Chicago violations: {chi_violations_parsed.count():,} -> {chi_violations_deduped.count():,} after dedup")

dallas_violations_deduped = dallas_violations_long.dropDuplicates(
    ["inspection_id", "violation_code", "violation_description"]
)
print(f"Dallas violations: {dallas_violations_long.count():,} -> {dallas_violations_deduped.count():,} after dedup")

# assignment: if score >= 90, keep only top 3 violations per inspection
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

w90 = Window.partitionBy("inspection_id").orderBy("violation_seq")

before_trim = dallas_violations_deduped.count()

dallas_violations_deduped = (
    dallas_violations_deduped
    .withColumn("rn", row_number().over(w90))
    .filter(
        (col("Inspection Score").cast("int") < 90) | (col("rn") <= 3)
    )
    .drop("rn")
)

after_trim = dallas_violations_deduped.count()
print(f"Dallas violations before score>=90 trim: {before_trim:,}")
print(f"Dallas violations after trim to top 3:   {after_trim:,}")
print(f"Trimmed: {before_trim - after_trim:,}")

Write Silver tables to Delta

In [0]:
from pyspark.sql.functions import trim, col

drop_cols = ["_errors", "_warnings"]

# Chicago inspections
chi_clean = chi_good
for c in drop_cols:
    if c in chi_clean.columns:
        chi_clean = chi_clean.drop(c)

chi_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .saveAsTable(f"{catalog_name}.{silver_schema}.chicago_inspections")
print(f"chicago_inspections written: {chi_clean.count():,} rows")

# Dallas inspections
dal_clean = dal_good
for c in drop_cols:
    if c in dal_clean.columns:
        dal_clean = dal_clean.drop(c)

dal_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .saveAsTable(f"{catalog_name}.{silver_schema}.dallas_inspections")
print(f"dallas_inspections written: {dal_clean.count():,} rows")

# Chicago violations (deduped)
chi_violations_deduped.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .saveAsTable(f"{catalog_name}.{silver_schema}.chicago_violations")
print(f"chicago_violations written: {chi_violations_deduped.count():,} rows")

# Dallas violations (deduped)
dallas_violations_deduped.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .saveAsTable(f"{catalog_name}.{silver_schema}.dallas_violations")
print(f"dallas_violations written: {dallas_violations_deduped.count():,} rows")

# Quarantine (only records with actual errors, not just warnings)
from pyspark.sql.functions import size

chi_errors_only = chi_bad.filter(size(col("_errors")) > 0)
chi_errors_count = chi_errors_only.count()
if chi_errors_count > 0:
    chi_errors_only.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("delta.columnMapping.mode", "name") \
        .option("delta.minReaderVersion", "2") \
        .option("delta.minWriterVersion", "5") \
        .saveAsTable(f"{catalog_name}.{silver_schema}.quarantine_chicago")
    print(f"quarantine_chicago written: {chi_errors_count:,} rows")
else:
    print("quarantine_chicago: no error records")

dal_errors_only = dal_bad.filter(size(col("_errors")) > 0)
dal_errors_count = dal_errors_only.count()
if dal_errors_count > 0:
    dal_errors_only.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("delta.columnMapping.mode", "name") \
        .option("delta.minReaderVersion", "2") \
        .option("delta.minWriterVersion", "5") \
        .saveAsTable(f"{catalog_name}.{silver_schema}.quarantine_dallas")
    print(f"quarantine_dallas written: {dal_errors_count:,} rows")
else:
    print("quarantine_dallas: no error records")

Verify Silver tables

In [0]:
print("=" * 60)
print("SILVER TABLES")
print("=" * 60)
spark.sql(f"SHOW TABLES IN {catalog_name}.{silver_schema}").show(truncate=False)

print("\nChicago Inspections sample:")
spark.sql(f"SELECT `Inspection ID`, `DBA Name`, Results, inspection_score, City, violation_count, has_urgent_critical FROM {catalog_name}.{silver_schema}.chicago_inspections LIMIT 5").show(truncate=False)

print("Dallas Inspections sample:")
spark.sql(f"SELECT inspection_id, `Restaurant Name`, `Inspection Date`, `Inspection Score`, `Zip Code`, violation_count, has_urgent_critical FROM {catalog_name}.{silver_schema}.dallas_inspections LIMIT 5").show(truncate=False)

print("Chicago Violations sample:")
spark.sql(f"SELECT * FROM {catalog_name}.{silver_schema}.chicago_violations LIMIT 5").show(truncate=False)

print("Dallas Violations sample (with extracted violation_code):")
spark.sql(f"SELECT inspection_id, violation_code, violation_description, violation_points FROM {catalog_name}.{silver_schema}.dallas_violations LIMIT 5").show(truncate=60)

Summary

In [0]:
print("=" * 60)
print("SILVER LAYER SUMMARY")
print("=" * 60)

chi_s = spark.table(f"{catalog_name}.{silver_schema}.chicago_inspections").count()
dal_s = spark.table(f"{catalog_name}.{silver_schema}.dallas_inspections").count()
chi_v = spark.table(f"{catalog_name}.{silver_schema}.chicago_violations").count()
dal_v = spark.table(f"{catalog_name}.{silver_schema}.dallas_violations").count()

print(f"Chicago inspections : {chi_s:,}")
print(f"Dallas inspections  : {dal_s:,}")
print(f"Chicago violations  : {chi_v:,} (deduped)")
print(f"Dallas violations   : {dal_v:,} (deduped, trimmed to 3 for score>=90)")
print(f"Chicago quarantined : {chi_errors_count:,}")
print(f"Dallas quarantined  : {dal_errors_count:,}")

print(f"""
Validation rules applied:
  Restaurant Name, Date, Type, Zip not null (both cities)
  Zip codes validated for correct format (both cities)
  Chicago Results not null
  Dallas score <= 100 and >= 0
  Every inspection must have >= 1 violation (error, dropped)
  Dallas score >= 90: violations trimmed to top 3 per inspection
  Result cannot be PASS with Urgent/Critical violations (error, dropped)
  Violations deduped by content (not slot position)
  City/State casing standardized
  Chicago scores derived from Results
  Dallas synthetic inspection_id generated via sha2 hash
  Dallas violation_code extracted from description
""")